In [1]:
from pathlib import Path
import shutil
import geopandas as gpd

In [7]:
def process_convert_state_shapefiles_to_geojson(state_abbr):
    downloads_dir = Path("./downloads")
    # geojson_dir = Path("./geojsons")
    # converted_dir = Path("./converted")

    state_dir = downloads_dir / state_abbr
    if not state_dir.is_dir():
        Exception("no state dir")
    state = state_dir.name

    before_dir = state_dir / "before"
    before_dir.mkdir(exist_ok=True)

    for item in state_dir.iterdir():
        if item.name == "before":
            continue

        shutil.move(str(item), str(before_dir / item.name))

    # Output folders
    (state_dir / 'converted').mkdir(parents=True, exist_ok=True)
    (state_dir / 'geojsons').mkdir(parents=True, exist_ok=True)

    # Each extracted dataset
    for dataset_dir in before_dir.iterdir():

        if not dataset_dir.is_dir():
            continue

        shp_files = list(dataset_dir.rglob("*.shp"))

        if not shp_files:
            print(f"No shapefiles in {dataset_dir}")
            continue

        success = True

        for shp_path in shp_files:
            try:
                print(f"Converting {shp_path}")

                gdf = gpd.read_file(shp_path)

                if gdf.crs and gdf.crs != "EPSG:4326":
                    gdf = gdf.to_crs("EPSG:4326")

                out_dir = state_dir / 'geojsons'
                out_dir.mkdir(parents=True, exist_ok=True)

                out_path = out_dir / f"{shp_path.stem}.geojson"
                gdf.to_file(out_path, driver="GeoJSON")

            except Exception as e:
                print(f"Failed: {shp_path}: {e}")
                success = False

        # Move the dataset folder if everything converted successfully
        if success:
            shutil.move(
                str(dataset_dir),
                str(state_dir / 'converted' / dataset_dir.name)
            )

In [29]:
process_convert_state_shapefiles_to_geojson("MO")

Converting downloads\MO\before\Missouri_CD_1992_2000\Missouri_CD_1992_2000.shp
Converting downloads\MO\before\MO_2012to2020_CD\MO.shp
Converting downloads\MO\before\MO_2012to2020_SL\MO_Leg_Lower_2012to2020.shp
Converting downloads\MO\before\MO_2012to2020_SU\MO_Leg_Upper_2012to2020.shp
Converting downloads\MO\before\MO_CD_2002_2010\CD_tl_2010_29_cd111.shp
Converting downloads\MO\before\MO_CD_Enacted05182022\MO_CD_Enacted_05182022.shp
Converting downloads\MO\before\MO_CD_MO_2025\MO_CD_MO_First_2025.shp
Converting downloads\MO\before\MO_LD_Enacted01192022\MO_HD_01202022.shp
Converting downloads\MO\before\mo_lower_1992to2000\mo_lower_1992to2000.shp
Converting downloads\MO\before\mo_lower_2002to2010\mo_lower_2002to2010.shp
Converting downloads\MO\before\MO_SD_Enacted03152022\MO_SD_Enacted_03152022.shp
Converting downloads\MO\before\mo_upper_1992to2000\mo_upper_1992to2000.shp
Converting downloads\MO\before\mo_upper_2002to2010\mo_upper_2002to2010.shp
